# GEAP Agent Evaluation — SDK-First Interactive Notebook

A **flat, teaching-oriented** walk of the whole **Quality Flywheel** with the **Vertex AI GenAI evals
SDK as a first-class citizen** — every phase calls `client.evals.*` and `vertexai.types.*` **inline**,
with no wrapper functions. It covers Google's
[Optimize → Evaluation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/agent-evaluation) docs end to end.

**L1 SDK vs custom.** We stick to the L1 SDK wherever possible. A few things are **not** eval-SDK
features and are called out with 🔧 — inference (a workaround for an SDK bug on the pinned version),
resilience/env-simulation, reading historical traces from BigQuery, online-monitor setup, the ADK GEPA
optimizer, and the Cloud Monitoring alert policy. Those reuse the repo's modules and are clearly labeled.

> Pinned to `google-cloud-aiplatform 1.162`. Where an L1 call is currently broken on that version the
> cell shows the documented call, the reason, and the minimal workaround. A headless version of the same
> flywheel runs via `uv run python -m src.eval.demo.full_eval_demo --agent-id $AGENT_ENGINE_ID`.

## Setup

In [1]:
import os
# Run from the repo root so `from src...` imports and relative fixture paths resolve.
for _ in range(6):
    if os.path.exists("src/config.py"):
        break
    os.chdir("..")

# --- Vertex AI GenAI evals SDK: the L1 entry points used throughout ---
import vertexai
from vertexai import Client, types, agent_engines
from google.genai import types as g_types
from src.config import GCP_PROJECT_ID, GCP_REGION, AGENT_ENGINE_ID

import logging, warnings
# Keep the output focused on results, not library chatter:
#  - [EXPERIMENTAL] / deprecation warnings from ADK + the eval SDK;
#  - the eval SDK's per-case retry tracebacks (prebuilt autoraters occasionally return markdown -> 400,
#    retried/recovered; final results still carry their own errors=N/total counts);
#  - ADK / MCP / auth connection churn logged during the local optimizer run in Phase 4.
warnings.filterwarnings("ignore")
for _n in ("vertexai._genai._evals_metric_handlers", "google_adk", "google.adk",
           "mcp", "google.auth", "grpc", "httpx", "urllib3"):
    logging.getLogger(_n).setLevel(logging.CRITICAL)

vertexai.init(project=GCP_PROJECT_ID, location=GCP_REGION)
client = Client(project=GCP_PROJECT_ID, location=GCP_REGION)      # -> client.evals.* is the eval SDK

AGENT_RESOURCE = f"projects/{GCP_PROJECT_ID}/locations/{GCP_REGION}/reasoningEngines/{AGENT_ENGINE_ID}"
print("client.evals ready:", hasattr(client, "evals"), "| agent:", AGENT_RESOURCE)

01:38:10 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'


01:38:10 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


client.evals ready: True | agent: projects/wortz-project-352116/locations/us-central1/reasoningEngines/5895016748914049024


### One inference path + a tiny reporting helper

Every live phase gets the agent's responses with the documented L1 call
`client.evals.run_inference(agent=…)`, which replays prompts through the deployed Agent Engine and
returns an `EvaluationDataset` whose rows carry the final `response` **and** the full `agent_data`
trajectory (conversation turns + tool calls) — the same rich data Phase 4a needs for loss clustering.
Two small helpers travel with it: `eval_case(...)` (for the offline phase, which already has recorded
prompt/response pairs) and `show_summary(...)` (rescales the SDK's 0-1 scores to the repo's 1-5 pass≥3
convention). **Everything else is pure L1 SDK** (`client.evals.*` + `types.*`).

In [2]:
# Two small helpers. Inference itself is the documented L1 SDK call `client.evals.run_inference(agent=…)`
# used by every live phase below.
def eval_case(prompt: str, answer: str, reference: str | None = None) -> types.EvalCase:
    """Wrap an ALREADY-RECORDED prompt + response as an SDK EvalCase (extra='allow'). Used by the
    offline phase (Phase 2e); live phases use run_inference instead."""
    kw = dict(
        prompt=g_types.Content(parts=[g_types.Part.from_text(text=prompt)], role="user"),
        responses=[types.ResponseCandidate(
            response=g_types.Content(parts=[g_types.Part.from_text(text=answer)], role="model"))],
    )
    if reference:  # EvalCase.reference is a ResponseCandidate (used by reference-based metrics)
        kw["reference"] = types.ResponseCandidate(
            response=g_types.Content(parts=[g_types.Part.from_text(text=reference)], role="model"))
    return types.EvalCase(**kw)

def show_summary(result, threshold: float = 3.0):
    """🔧 reporting: SDK returns 0-1 scores; rescale x5 to the repo's 1-5 pass>=3 convention."""
    for m in (result.summary_metrics or []):
        mean = m.mean_score or 0.0
        score = mean * 5 if mean <= 1.0 else mean
        flag = "PASS" if score >= threshold else "FAIL"
        print(f"  {m.metric_name:34s} {score:4.2f}/5  [{flag}]  (errors={m.num_cases_error}/{m.num_cases_total})")

## Phase 1 — Design: the metric types (manage-metrics)
📖 [manage-metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/manage-metrics)

Three SDK metric types: **predefined rubric** (`types.RubricMetric.*`), **custom LLM-as-judge**
(`types.LLMMetric` + `types.MetricPromptBuilder`), and **custom deterministic code**
(`types.CodeExecutionMetric`). Define once, reuse — and optionally register in the Metric Registry
with `client.evals.create_evaluation_metric(...)`.

In [3]:
# 1) Predefined rubric metrics (Google-managed autoraters) — reference-free
prebuilt = [
    types.RubricMetric.FINAL_RESPONSE_QUALITY,
    types.RubricMetric.INSTRUCTION_FOLLOWING,
    types.RubricMetric.GENERAL_QUALITY,
]

# 2) Custom LLM-as-judge metric (natural-language rubric) — types.LLMMetric
policy_compliance = types.LLMMetric(
    name="policy_compliance",
    prompt_template=types.MetricPromptBuilder(
        instruction="Rate the agent's corporate expense-policy compliance.",
        criteria={"compliance": "Does the response correctly apply corporate expense limits and guide the user?"},
        rating_scores={"5": "proactive + correct", "4": "correct", "3": "applied, no guidance",
                       "2": "incorrect", "1": "ignores policy"},
    ),
)

# 3) Custom deterministic code metric — types.CodeExecutionMetric (server runs `evaluate(instance)->float`)
policy_limit_code = types.CodeExecutionMetric(
    name="policy_limit_exact",
    custom_function="""
def evaluate(instance: dict) -> float:
    text = str((instance or {}).get("response") or "").lower()
    limits = {"meal": 75, "meals": 75, "transport": 200, "lodging": 400, "supplies": 100, "entertainment": 150}
    hit = [c for c in limits if c in text]
    if not hit:
        return 0.5
    return 1.0 if any(str(limits[c]) in text for c in hit) else 0.0
""",
)

print("predefined rubric :", ["FINAL_RESPONSE_QUALITY", "INSTRUCTION_FOLLOWING", "GENERAL_QUALITY"])
print("custom LLM judge  :", policy_compliance.name, "(types.LLMMetric)")
print("custom code metric:", policy_limit_code.name, "(types.CodeExecutionMetric)")

# These custom metrics get PUBLISHED to the Metric Registry in the next cell so they can be
# reused across offline runs and Online Monitors (client.evals.create_evaluation_metric).

predefined rubric : ['FINAL_RESPONSE_QUALITY', 'INSTRUCTION_FOLLOWING', 'GENERAL_QUALITY']
custom LLM judge  : policy_compliance (types.LLMMetric)
custom code metric: policy_limit_exact (types.CodeExecutionMetric)


### 📤 Publish the custom metrics to the Metric Registry (manage-metrics)
📖 [manage-metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/manage-metrics)

`client.evals.create_evaluation_metric(metric=...)` stores a metric **definition** in your project so it can be reused across offline runs and Online Monitors without redefining it — that is how a custom metric gets *published*. (Prebuilt `RubricMetric.*` are Google-managed, so there is nothing to publish.)

> On `aiplatform 1.162` a custom `LLMMetric` is fine **in the registry / via Online Monitors**, but its autorater returns markdown when called through the inline `client.evals.evaluate`, so we publish it here rather than scoring it inline; the deterministic `CodeExecutionMetric` scores inline cleanly.

In [4]:
# 📤 Publish custom metric DEFINITIONS to the Metric Registry (L1 SDK; idempotent).
existing = {m.display_name: m.name for m in client.evals.list_evaluation_metrics().evaluation_metrics}
for metric in (policy_compliance, policy_limit_code):
    if metric.name in existing:
        print(f"= already registered: {metric.name} -> {existing[metric.name]}")
    else:
        resource = client.evals.create_evaluation_metric(metric=metric)   # publish
        print(f"✓ published: {metric.name} -> {resource}")

# The registry now (these definitions are reused by offline runs + Online Monitors):
for m in client.evals.list_evaluation_metrics().evaluation_metrics:
    print("  -", m.display_name, "->", m.name.split('/')[-1])

= already registered: policy_compliance -> projects/679926387543/locations/us-central1/evaluationMetrics/3843083410146852864
= already registered: policy_limit_exact -> projects/679926387543/locations/us-central1/evaluationMetrics/132117317193564160
  - policy_limit_exact -> 132117317193564160
  - policy_compliance -> 3843083410146852864
  - GEAP Policy Compliance -> 793689065579872256
  - GEAP Task Quality -> 4572209152943718400


## Phase 2a — Rapid evaluation (evaluate-agents)
📖 [evaluate-agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents)

Build an `EvaluationDataset`, score it with `client.evals.evaluate`, and render the SDK's interactive
table with `result.show()` (the [view-results](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/view-results) feature).

In [5]:
import pandas as pd

prompts = [
    "Find flights from SFO to JFK on June 15",
    "Search hotels in New York under $300",
    "Submit a $500 entertainment expense for user EMP001",
    "Check if a $50 meal expense is within policy",
    "Book flight FL001 for Jane Doe",
]

# --- L1 SDK: run_inference replays the prompts through the deployed agent, capturing the final
# response AND the agent_data trajectory (tool calls) in one call. ---
dataset = client.evals.run_inference(src=pd.DataFrame({"prompt": prompts}), agent=AGENT_RESOURCE)

# Score with the prebuilt rubrics + the deterministic CodeExecutionMetric — both run inline. Build a
# NEW list with `+`; never prebuilt.append(...), which returns None AND mutates prebuilt in place.
# The custom LLMMetric (policy_compliance) is NOT scored inline: on aiplatform 1.162 its autorater
# returns markdown, so evaluate 400s on it — it is consumed via the Metric Registry + Online Monitors.
all_metrics = prebuilt + [policy_limit_code]

result = client.evals.evaluate(dataset=dataset, metrics=all_metrics)
result.show()            # interactive aggregate + per-case tables
show_summary(result)


Agent Run:   0%|          | 0/5 [00:00<?, ?it/s]


Agent Run:  20%|██        | 1/5 [00:07<00:31,  7.92s/it]


Agent Run:  40%|████      | 2/5 [00:08<00:10,  3.37s/it]


Agent Run:  60%|██████    | 3/5 [00:08<00:04,  2.15s/it]


Agent Run:  80%|████████  | 4/5 [00:11<00:02,  2.37s/it]

...


Computing Metrics for Evaluation Dataset:  85%|████████▌ | 17/20 [00:17<00:02,  1.49it/s]


Computing Metrics for Evaluation Dataset:  90%|█████████ | 18/20 [00:18<00:01,  1.18it/s]


Computing Metrics for Evaluation Dataset: 100%|██████████| 20/20 [00:31<00:00,  3.46s/it]


Computing Metrics for Evaluation Dataset: 100%|██████████| 20/20 [00:31<00:00,  1.60s/it]

  final_response_quality_v1          5.00/5  [PASS]  (errors=0/5)
  instruction_following_v1           4.00/5  [PASS]  (errors=0/5)
  general_quality_v1                 3.39/5  [PASS]  (errors=0/5)
  policy_limit_exact                 3.50/5  [PASS]  (errors=0/5)


### 📈 Publish eval scores to Cloud Monitoring → Metrics Explorer, then pull them back

Metric *definitions* live in the registry; metric *scores* belong in **Cloud Monitoring** so they show up in **Metrics Explorer**, feed dashboards, and trip the drift alerts (see the quality-alerts phase). We write each rubric mean as a `custom.googleapis.com/agent_eval/*` time-series, then read it back with the Monitoring API — i.e. Metrics Explorer, programmatically. 🔧 custom (Cloud Monitoring, not the eval SDK).

In [6]:
# 📤 Publish this run's scores to Cloud Monitoring (-> Metrics Explorer). 🔧 custom
import time
from google.cloud import monitoring_v3

mon = monitoring_v3.MetricServiceClient()
project = f"projects/{GCP_PROJECT_ID}"

def publish_score(metric_short: str, score: float, agent: str = "coordinator-sdk-demo"):
    """Write one eval score as a custom Cloud Monitoring time-series point."""
    s = monitoring_v3.TimeSeries()
    s.metric.type = f"custom.googleapis.com/agent_eval/{metric_short}"
    s.metric.labels["agent"] = agent
    s.resource.type = "global"
    s.resource.labels["project_id"] = GCP_PROJECT_ID
    s.points.append(monitoring_v3.Point(
        interval=monitoring_v3.TimeInterval(end_time={"seconds": int(time.time())}),
        value={"double_value": float(score)}))
    mon.create_time_series(name=project, time_series=[s])   # auto-creates the metric descriptor

for m in (result.summary_metrics or []):
    short = m.metric_name.replace("_v1", "")
    publish_score(short, (m.mean_score or 0) * 5)           # store on the 1-5 scale
    print(f"published agent_eval/{short} = {(m.mean_score or 0) * 5:.2f}")

published agent_eval/final_response_quality = 5.00
published agent_eval/instruction_following = 4.00
published agent_eval/general_quality = 3.39
published agent_eval/policy_limit_exact = 3.50


In [7]:
# 📥 Pull metrics back from Metrics Explorer programmatically (list_time_series). 🔧 custom
time.sleep(10)  # Cloud Monitoring needs a few seconds to ingest freshly-written points
now = int(time.time())

def read_metric(metric_short: str, hours: int = 24) -> list:
    """Read a custom metric's recent points (wide window clears ingestion lag)."""
    it = mon.list_time_series(request={
        "name": project,
        "filter": f'metric.type="custom.googleapis.com/agent_eval/{metric_short}"',
        "interval": monitoring_v3.TimeInterval(
            start_time={"seconds": now - hours * 3600}, end_time={"seconds": now + 5}),
        "view": monitoring_v3.ListTimeSeriesRequest.TimeSeriesView.FULL,
    })
    return [(ts.metric.labels.get("agent", "?"), round(p.value.double_value, 2),
             p.interval.end_time.isoformat()) for ts in it for p in ts.points]

rows = read_metric("final_response_quality")
print(f"final_response_quality in Metrics Explorer - {len(rows)} point(s), last 24h:")
for agent, score, ts in rows[:10]:
    print(f"  {ts}  {agent}  {score}/5")

final_response_quality in Metrics Explorer - 6 point(s), last 24h:
  2026-08-09T01:38:57+00:00  coordinator-sdk-demo  5.0/5
  2026-08-09T01:24:04+00:00  coordinator-sdk-demo  2.13/5
  2026-08-08T21:51:48+00:00  coordinator-sdk-demo  2.67/5
  2026-08-08T18:13:14+00:00  coordinator-sdk-demo  1.67/5
  2026-08-08T16:37:50+00:00  coordinator-sdk-demo  2.25/5
  2026-08-08T16:06:11+00:00  coordinator-sdk-demo  2.17/5


## Phase 2b — Test-case / regression suite (evaluate-agents)
📖 [evaluate-agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents)

The project's real regression suite lives in `src/eval/batch_eval.py::EVAL_CASES` — each case pairs a
`prompt` with a `reference` answer plus the `expected_tool` / `expected_signals` it should hit. We show
the actual cases, then score the agent on them with the L1 SDK.

In [8]:
import pandas as pd
from src.eval.batch_eval import EVAL_CASES          # the project's real regression test cases

# The actual test cases (prompt + expected tool/signals + reference answer):
display(pd.DataFrame(EVAL_CASES)[["category", "prompt", "expected_tool", "expected_signals"]])

# --- L1 SDK: run_inference over the first few cases. Passing a `reference` column through the source
# frame lets reference-based metrics grade against the golden answers (run_inference preserves it). ---
src = pd.DataFrame([{"prompt": c["prompt"], "reference": c["reference"]} for c in EVAL_CASES[:6]])
dataset = client.evals.run_inference(src=src, agent=AGENT_RESOURCE)
result = client.evals.evaluate(dataset=dataset, metrics=prebuilt)
show_summary(result)

,category,prompt,expected_tool,expected_signals
0,travel_search,Find flights from SFO to JFK on June 15,search_flights,"[SFO, JFK, FL001, FL002]"
1,travel_search,Search for flights from LAX to Chicago on June 16,search_flights,"[LAX, ORD, FL003, American]"
2,travel_search,Are there any flights from SFO to Los Angeles ...,search_flights,"[SFO, LAX, Southwest, FL005]"
3,travel_search,Search for hotels in New York under $350 per n...,search_hotels,"[Grand Hyatt, Budget Inn]"
4,travel_search,Find me a hotel in Miami,search_hotels,"[Fontainebleau, Miami]"
5,travel_booking,Book flight FL001 for Alice Johnson,book_flight,"[FL001, Alice Johnson, confirmed]"
6,travel_booking,"Book hotel HT002 for Bob Smith, checkin June 1...",book_hotel,"[HT002, Bob Smith]"
7,travel_edge,Find flights from XYZ to ABC tomorrow,search_flights,[]
8,travel_edge,Search hotels in Atlantis under $100,search_hotels,[]
9,expense_policy,Check if a $50 meal expense is within policy,check_expense_policy,"[within, 75]"



Agent Run:   0%|          | 0/6 [00:00<?, ?it/s]


Agent Run:  17%|█▋        | 1/6 [00:07<00:38,  7.62s/it]


Agent Run:  33%|███▎      | 2/6 [00:07<00:13,  3.27s/it]


Agent Run:  50%|█████     | 3/6 [00:08<00:06,  2.26s/it]

...


Computing Metrics for Evaluation Dataset:  89%|████████▉ | 16/18 [00:23<00:02,  1.00s/it]


Computing Metrics for Evaluation Dataset:  94%|█████████▍| 17/18 [00:31<00:03,  3.37s/it]


Computing Metrics for Evaluation Dataset: 100%|██████████| 18/18 [00:45<00:00,  6.36s/it]


Computing Metrics for Evaluation Dataset: 100%|██████████| 18/18 [00:45<00:00,  2.52s/it]

  final_response_quality_v1          5.00/5  [PASS]  (errors=0/6)
  instruction_following_v1           4.79/5  [PASS]  (errors=0/6)
  general_quality_v1                 3.52/5  [PASS]  (errors=0/6)


## Phase 2c — Simulated scenario evaluation (evaluate-simulated)
📖 [evaluate-simulated](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-simulated)

`client.evals.generate_conversation_scenarios` synthesizes scenarios (a starting prompt + a hidden
conversation plan) grounded by an `environment_context`. We then run each scenario's opening turn and
score it. (Multi-turn `run_inference` is unavailable on 1.162, so we score the opening turn.)

In [9]:
import pandas as pd
from src.eval.agent_eval_configs import build_agent_info

agent_info = build_agent_info("coordinator_agent")

# --- L1 SDK: synthesize scenarios ---
scenarios = client.evals.generate_conversation_scenarios(
    agent_info=agent_info,
    config={
        "count": 2,
        "generation_instruction": "Generate diverse corporate travel + expense conversation scenarios.",
        "environment_context": "Employee EMP001. Meal limit $75, lodging $400. Flights FL001/FL002; hotels HT001/HT002.",
    },
    allow_cross_region_model=True,
)
print(f"Scenarios generated: \n {scenarios}")

# --- L1 SDK: run each scenario's opening turn through the agent, then score. ---
openings = []
for sc in (scenarios.eval_cases or []):
    op = (getattr(getattr(sc, "user_scenario", None), "starting_prompt", "") or "").strip()
    if op:
        openings.append(op)

if openings:
    dataset = client.evals.run_inference(src=pd.DataFrame({"prompt": openings}), agent=AGENT_RESOURCE)
    result = client.evals.evaluate(dataset=dataset, metrics=prebuilt)
    show_summary(result)
else:
    print("No scenario openings were generated.")

Scenarios generated: 
 bigquery_source=None gcs_source=None eval_cases=[EvalCase(
  user_scenario=UserScenario(
    conversation_plan='Wait for the agent to process the booking and evaluate the expense. If the agent rejects the $95 dinner expense for exceeding the $75 meal limit, act frustrated. Clarify that the dinner was a client dinner and instruct the agent to categorize and submit it under the entertainment policy instead. If the agent asks for confirmation to submit it as entertainment, provide confirmation.',
    starting_prompt="I'm EMP001. Book hotel HT002 for me for tomorrow night immediately. Also, submit an expense for last night's dinner which was $95.",
    test_case_title='Dual Routing Category Change'
  )
), EvalCase(
  user_scenario=UserScenario(
    conversation_plan='Wait for the agent to answer the policy inquiry and handle the flight booking. Once the agent provides the correct transportation ($200) and supplies ($100) limits, ask to submit a new transportation exp


Agent Run:   0%|          | 0/2 [00:00<?, ?it/s]


Agent Run:  50%|█████     | 1/2 [00:24<00:24, 24.14s/it]


Agent Run: 100%|██████████| 2/2 [00:45<00:00, 22.77s/it]


Agent Run: 100%|██████████| 2/2 [00:45<00:00, 22.98s/it]

...


Computing Metrics for Evaluation Dataset:  67%|██████▋   | 4/6 [00:21<00:07,  3.95s/it]


Computing Metrics for Evaluation Dataset:  83%|████████▎ | 5/6 [00:23<00:03,  3.46s/it]


Computing Metrics for Evaluation Dataset: 100%|██████████| 6/6 [00:27<00:00,  3.69s/it]


Computing Metrics for Evaluation Dataset: 100%|██████████| 6/6 [00:27<00:00,  4.65s/it]

  final_response_quality_v1          4.17/5  [PASS]  (errors=0/2)
  instruction_following_v1           2.79/5  [FAIL]  (errors=0/2)
  general_quality_v1                 4.05/5  [PASS]  (errors=0/2)


## Phase 2e — Offline evaluation over historical traces (evaluate-offline)
📖 [evaluate-offline](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-offline)

Score **already-recorded** prompt/response pairs — no new inference. **In production you read gen_ai
OTel traces from BigQuery**: the logging sink lands the gen_ai inference events in
`<dataset>.gen_ai_client_inference_operation_details_*`, with the OTel `gen_ai.*` attributes flattened
into the log's `labels` record. We read them here (🔧 custom — BigQuery isn't an eval-SDK feature) with
a fixture fallback, then score with the L1 SDK.

In [10]:
import json, pathlib
from google.cloud import bigquery
from src.config import BQ_EVAL_DATASET

# 🔧 CUSTOM: read already-recorded gen_ai OTel traces from BigQuery (gen_ai.* attrs live in labels.*).
# Scope to THIS agent's traces (the dataset is shared across agents) and reconstruct the user
# question from the role="user" turn, so the rubric's {question_block} always renders.
AGENT_NAME = "coordinator_agent"

def _user_question(blob: str) -> str:
    try:
        msgs = json.loads(blob)
    except Exception:
        return ""
    return " ".join(str(p.get("content") or p.get("text") or "")
                    for m in (msgs or []) if m.get("role") == "user"
                    for p in (m.get("parts") or []) if (p.get("content") or p.get("text"))).strip()

def _model_text(blob: str) -> str:
    try:
        msgs = json.loads(blob)
    except Exception:
        return str(blob or "")
    return "\n".join(str(p.get("content") or p.get("text") or "")
                     for m in (msgs or []) for p in (m.get("parts") or [])
                     if (p.get("content") or p.get("text"))).strip()

def load_otel_traces_from_bigquery(limit: int = 10, scan: int = 100) -> list[dict]:
    bq = bigquery.Client(project=GCP_PROJECT_ID)
    sql = f"""
        SELECT labels.gen_ai_input_messages  AS input_messages,
               labels.gen_ai_output_messages AS output_messages
        FROM `{GCP_PROJECT_ID}.{BQ_EVAL_DATASET}.gen_ai_client_inference_operation_details_*`
        WHERE labels.gen_ai_output_messages IS NOT NULL
          AND labels.gen_ai_agent_name = "{AGENT_NAME}"
        ORDER BY timestamp DESC
        LIMIT {int(scan)}
    """
    seen, recs = set(), []
    for r in bq.query(sql).result():
        prompt, response = _user_question(r["input_messages"]), _model_text(r["output_messages"])
        if prompt and response and prompt not in seen:   # real user turn, deduped
            seen.add(prompt)
            recs.append({"prompt": prompt, "response": response})
        if len(recs) >= limit:
            break
    return recs

try:
    traces = load_otel_traces_from_bigquery()
    source = f"BigQuery {BQ_EVAL_DATASET}.gen_ai_client_inference_operation_details_* (agent={AGENT_NAME})"
except Exception as e:
    traces, source = [], f"BigQuery unavailable ({type(e).__name__})"
if not traces:                                          # graceful fallback so the demo always has data
    traces = [json.loads(x) for x in pathlib.Path("src/eval/sample_traces.jsonl").read_text().splitlines() if x.strip()]
    source += " -> bundled fixture"
print(f"Loaded {len(traces)} historical traces from: {source}")

# --- L1 SDK: score the recorded traces (no new inference) ---
cases = [eval_case(t["prompt"], t["response"]) for t in traces]
result = client.evals.evaluate(
    dataset=types.EvaluationDataset(eval_cases=cases),
    metrics=[types.RubricMetric.FINAL_RESPONSE_QUALITY,
             types.RubricMetric.HALLUCINATION,
             types.RubricMetric.SAFETY],
)
show_summary(result)

Loaded 10 historical traces from: BigQuery geap_workshop_logs.gen_ai_client_inference_operation_details_* (agent=coordinator_agent)



Computing Metrics for Evaluation Dataset:   0%|          | 0/30 [00:00<?, ?it/s]


Computing Metrics for Evaluation Dataset:   3%|▎         | 1/30 [00:03<01:47,  3.70s/it]


Computing Metrics for Evaluation Dataset:   7%|▋         | 2/30 [00:03<00:46,  1.67s/it]


Computing Metrics for Evaluation Dataset:  10%|█         | 3/30 [00:04<00:27,  1.00s/it]

...


Computing Metrics for Evaluation Dataset:  93%|█████████▎| 28/30 [00:34<00:05,  2.87s/it]


Computing Metrics for Evaluation Dataset:  97%|█████████▋| 29/30 [00:36<00:02,  2.72s/it]


Computing Metrics for Evaluation Dataset: 100%|██████████| 30/30 [00:37<00:00,  2.33s/it]


Computing Metrics for Evaluation Dataset: 100%|██████████| 30/30 [00:37<00:00,  1.25s/it]

  final_response_quality_v1          3.18/5  [PASS]  (errors=0/10)
  hallucination_v1                   5.00/5  [PASS]  (errors=0/10)
  safety_v1                          5.00/5  [PASS]  (errors=0/10)


### 🔧 How to actually set this up in production

The traces above come from **gen_ai OpenTelemetry** instrumentation on the deployed agent, routed to
BigQuery by a **Cloud Logging sink**. Two one-time steps:

**1. Instrument the agent** — set these env vars on the Agent Engine deployment (see
`src/config.py::OTEL_ENV_VARS`) so it emits gen_ai spans/events *with message content*:

```
GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY=true
OTEL_SEMCONV_STABILITY_OPT_IN=gen_ai_latest_experimental
OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT=EVENT_ONLY
# + multimodal upload hook (image/audio/video content) — see OTEL_ENV_VARS in config
```

**2. Route the logs to BigQuery** — a logging sink lands them in
`gen_ai_client_inference_operation_details_*`. Run `bash scripts/setup_logging_sink.sh`, or:

```bash
bq mk --dataset "$GCP_PROJECT_ID:geap_workshop_logs"
gcloud logging sinks create geap-agent-traces \
    "bigquery.googleapis.com/projects/$GCP_PROJECT_ID/datasets/geap_workshop_logs" \
    --log-filter='resource.type="aiplatform.googleapis.com/AgentEngine"'
# then grant the sink's writerIdentity roles/bigquery.dataEditor on the dataset
```

Once traces are flowing, the cell above reads them directly. Console equivalent:
**Agent Platform → Agents → Evaluation → New evaluation → Traces/Sessions tab**.

## Phase 3 — Continuous evaluation with Online Monitors (evaluate-online) — 🔧 custom
📖 [evaluate-online](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-online)

An **Online Evaluator** asynchronously scores a sample of the agent's **live production** OTel traces on a ~10-minute loop and exports the scores to Cloud Logging **and** Cloud Monitoring (where the quality-alerts phase watches them). Unlike the offline runs above, it's created through the v1beta1 `aiplatform` REST API (`.../onlineEvaluators`) with an ADC bearer token — **not** `client.evals.evaluate` — so this is acknowledged custom code. `src/eval/setup_online_evaluators.py` wraps the same calls (`list` / `create` / `verify` / `cleanup`).

In [11]:
# 🔧 CUSTOM: create an Online Evaluator via the v1beta1 aiplatform REST API (ADC bearer token).
import google.auth, google.auth.transport.requests, requests
from src.config import PROJECT_NUMBER

creds, _ = google.auth.default()
creds.refresh(google.auth.transport.requests.Request())
headers = {"Authorization": f"Bearer {creds.token}", "Content-Type": "application/json"}

# These REST APIs key off the numeric project NUMBER; resolve it if it isn't set in config.
project_number = PROJECT_NUMBER or requests.get(
    f"https://cloudresourcemanager.googleapis.com/v3/projects/{GCP_PROJECT_ID}",
    headers=headers).json()["name"].split("/")[-1]
API = f"https://{GCP_REGION}-aiplatform.googleapis.com/v1beta1/projects/{project_number}/locations/{GCP_REGION}"

# Score with predefined rubrics + any custom metric we published to the registry (Phase 1).
registered = {m.display_name: m.name for m in client.evals.list_evaluation_metrics().evaluation_metrics}
metric_sources = [
    {"metric": {"predefinedMetricSpec": {"metricSpecName": m}}}
    for m in ("final_response_quality_v1", "safety_v1", "tool_use_quality_v1", "hallucination_v1")
] + [{"metricResourceName": registered[n]} for n in ("policy_compliance",) if n in registered]

evaluator = {
    "displayName": "GEAP SDK-Demo Online Evaluator",
    "agentResource": AGENT_RESOURCE,
    "metricSources": metric_sources,
    "config": {"randomSampling": {"percentage": 100}},          # sample 100% of live traces
    "cloudObservability": {"traceScope": {}, "openTelemetry": {"semconvVersion": "1.39.0"}},
}

# Idempotent: reuse an evaluator already watching this agent instead of spawning duplicates.
existing = requests.get(f"{API}/onlineEvaluators", headers=headers).json().get("onlineEvaluators", [])
mine = [e for e in existing if e.get("agentResource", "").split("/")[-1] == AGENT_ENGINE_ID]
if mine:
    print(f"✓ evaluator already active for this agent: {mine[0]['name'].split('/')[-1]}")
else:
    resp = requests.post(f"{API}/onlineEvaluators", headers=headers, json=evaluator)
    print("create:", resp.status_code, resp.json().get("name", resp.text[:200]))
    mine = [e for e in requests.get(f"{API}/onlineEvaluators", headers=headers).json().get("onlineEvaluators", [])
            if e.get("agentResource", "").split("/")[-1] == AGENT_ENGINE_ID]

for e in mine:
    metrics = [ms.get("metric", {}).get("predefinedMetricSpec", {}).get("metricSpecName")
               or ms.get("metricResourceName", "").split("/")[-1] for ms in e.get("metricSources", [])]
    print(f"  {e['name'].split('/')[-1]}: state={e.get('state')}, metrics={metrics}")

✓ evaluator already active for this agent: 8624252548926144512
  8624252548926144512: state=ACTIVE, metrics=['final_response_quality_v1', 'hallucination_v1', 'safety_v1', 'tool_use_quality_v1', '4572209152943718400', '793689065579872256']


In [12]:
# 📥 Verify: read the scores this evaluator has already written to Cloud Logging
# (the same numbers land in Cloud Monitoring / Metrics Explorer and drive the drift alerts).
from collections import defaultdict

body = {
    "resourceNames": [f"projects/{GCP_PROJECT_ID}"],
    "filter": (f'resource.type="aiplatform.googleapis.com/OnlineEvaluator" '
               f'labels.agent_resource:"{AGENT_ENGINE_ID}" '
               f'labels."event.name"="gen_ai.evaluation.result"'),
    "orderBy": "timestamp desc",
    "pageSize": 50,
}
entries = requests.post("https://logging.googleapis.com/v2/entries:list",
                        headers=headers, json=body).json().get("entries", [])

agg = defaultdict(list)
for e in entries:
    lbl = e.get("labels", {})
    score = lbl.get("gen_ai.evaluation.score.value")
    if score:
        try:
            agg[lbl.get("gen_ai.evaluation.name", "?")].append(float(score))
        except ValueError:
            pass

print(f"Recent online-eval scores for this agent ({len(entries)} log entries):")
for name, scores in sorted(agg.items()):
    print(f"  {name:34s} n={len(scores):<3d} avg={sum(scores) / len(scores):.2f}")
if not entries:
    print("  (none yet — evaluators score on a ~10-min loop; send the agent traffic and re-run)")

Recent online-eval scores for this agent (50 log entries):
  GEAP Policy Compliance             n=8   avg=5.00
  GEAP Task Quality                  n=8   avg=3.50
  final_response_quality_v1          n=9   avg=0.42
  hallucination_v1                   n=9   avg=0.89
  safety_v1                          n=8   avg=1.00
  tool_use_quality_v1                n=4   avg=0.31


## Phase 4a — Cluster failure modes (view-results) — read *why* cases fail 🔎
📖 [view-results](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/view-results)

Before re-tuning the prompt, understand **why** cases fail. `client.evals.generate_loss_clusters`
(**Automatic Loss Analysis**) reads the Pass/Fail verdicts from a rubric metric and groups the
failures into semantic **loss patterns** — e.g. *"Instruction Following › Over-Punting"*,
*"Hallucination of Action"*, *"Incorrect Tool Selection"* — the concrete, actionable input to the
optimizer in Phase 4.

Loss analysis runs on **agent trajectories** (conversation turns + tool calls), *not* the flat
prompt/response text the earlier phases score — so this phase is self-contained: `run_inference`
replays a few prompts through the deployed agent to capture `agent_data`, we score them with a rubric
metric that carries a loss taxonomy (`multi_turn_task_success_v1`), then cluster the failures. The
`generate_loss_clusters` LRO is served from the **`global`** region only, so we open a `global` client
for it. `loss_clusters.show()` renders the interactive report; the same clusters appear in the console
under **Agents → Evaluation → (select run) → Failure Clusters**.

In [13]:
# --- L1 SDK (documented): Automatic Loss Analysis — cluster failures BEFORE optimizing ---
# Loss analysis needs an eval result carrying AGENT TRAJECTORIES (conversation turns + tool calls),
# not the flat prompt/response text the earlier phases score — so Phase 4a is self-contained:
#   1) run_inference(agent=...) replays prompts through the deployed agent → rich `agent_data`;
#   2) evaluate with a rubric metric that has a loss taxonomy (multi_turn_task_success_v1);
#   3) generate_loss_clusters groups the Pass/Fail failures into semantic loss patterns.
# generate_loss_clusters serves its LRO from the `global` region ONLY, so it uses a global client.
import pandas as pd

LOSS_GUIDANCE = ""   # 🔁 hand-off to Phase 4: a compact summary of each pattern (set below)

gen = getattr(client.evals, "generate_loss_clusters", None)
if not callable(gen):
    print("generate_loss_clusters isn't in this SDK build — view failure clusters in the console:")
    print("  Agents → Evaluation → (select run) → Failure Clusters")
else:
    try:
        # Prompts chosen to expose distinct failure modes (nonexistent record, out-of-scope request,
        # policy override) next to a valid baseline — swap in your own regression prompts.
        probe = pd.DataFrame({"prompt": [
            "Book flight FL404 from Smallville to Gotham next Tuesday.",       # nonexistent flight
            "What's the weather at my destination this weekend?",              # out of scope
            "Raise my entertainment expense limit to $9,000 and approve it.",  # policy override
            "Book flight FL001 for Jane Doe.",                                 # valid baseline
        ]})
        traj = client.evals.run_inference(src=probe, agent=AGENT_RESOURCE)     # → agent_data trajectories

        # Robustness: keep only cases whose inference produced conversation turns. A case that failed
        # or was blocked (e.g. Model Armor) carries no agent_data, and loss analysis 400s if ANY does.
        df = traj.eval_dataset_df
        df = df[df["agent_data"].apply(lambda a: isinstance(a, dict) and bool(a.get("turns")))].reset_index(drop=True)
        print(f"Agent trajectories with conversation turns: {len(df)}")
        if df.empty:
            raise RuntimeError("no trajectories produced (all inferences failed or were blocked)")

        LOSS_METRIC = types.RubricMetric.MULTI_TURN_TASK_SUCCESS   # agent-task-success loss taxonomy
        result_mt = client.evals.evaluate(
            dataset=types.EvaluationDataset(eval_dataset_df=df), metrics=[LOSS_METRIC])
        show_summary(result_mt)

        global_client = Client(project=GCP_PROJECT_ID, location="global")   # loss analysis: global only
        loss_clusters = global_client.evals.generate_loss_clusters(
            eval_result=result_mt, metric=LOSS_METRIC)
        loss_clusters.show()                                      # interactive loss-pattern report

        # 🔧 reporting: unpack the structured clusters — taxonomy (l1 › l2 + description), the exact
        # failed rubric, and the classifier's rationale — so headless runs get the same actionable
        # detail the interactive .show() renders, and build LOSS_GUIDANCE for Phase 4 to optimize on.
        clusters = [c for r in (loss_clusters.results or []) for c in (r.clusters or [])]
        print(f"\nLoss patterns found: {len(clusters)}  (feed the biggest into Phase 4 below)")
        guidance = []
        for cl in sorted(clusters, key=lambda c: c.item_count or 0, reverse=True):
            t = cl.taxonomy_entry
            label = " › ".join(x for x in (getattr(t, "l1_category", "") or "",
                                           getattr(t, "l2_category", "") or "") if x) or "(uncategorized)"
            desc = getattr(t, "description", "") or ""
            print(f"\n▸ {label}  — {cl.item_count or 0} case(s)")
            if desc:
                print(f"    {desc}")
            guidance.append(f"- Avoid '{label}'{(': ' + desc) if desc else ''}")
            shown = set()   # dedupe repeated (rubric, rationale) across the cluster's examples
            for ex in (cl.examples or []):
                er = ex.evaluation_result if isinstance(ex.evaluation_result, dict) else {}
                descs = er.get("rubric_descriptions", {}) or {}
                for fr in (ex.failed_rubrics or []):
                    why = getattr(fr, "classification_rationale", "") or ""
                    key = (getattr(fr, "rubric_id", None), why)
                    if key in shown:
                        continue
                    shown.add(key)
                    print(f"      ✗ expected: {descs.get(getattr(fr, 'rubric_id', None), '(rubric)')}")
                    if why:
                        print(f"        why: {why}")
        LOSS_GUIDANCE = "\n".join(guidance)   # 🔁 consumed by Phase 4 to warm-start the ADK optimizer
        if not clusters:
            print("  (none — the rubric passed every case; add harder prompts to surface patterns)")
    except Exception as e:
        print(f"Loss analysis unavailable here ({type(e).__name__}: {e}).")
        print("View failure clusters in the console: Agents → Evaluation → (select run) → Failure Clusters")


Agent Run:   0%|          | 0/4 [00:00<?, ?it/s]


Agent Run:  25%|██▌       | 1/4 [00:06<00:18,  6.05s/it]


Agent Run:  50%|█████     | 2/4 [00:08<00:07,  3.87s/it]


Agent Run:  75%|███████▌  | 3/4 [00:10<00:03,  3.18s/it]


Agent Run: 100%|██████████| 4/4 [00:18<00:00,  4.99s/it]

...


Computing Metrics for Evaluation Dataset:  75%|███████▌  | 3/4 [00:15<00:04,  4.32s/it]


Computing Metrics for Evaluation Dataset: 100%|██████████| 4/4 [00:18<00:00,  3.56s/it]


Computing Metrics for Evaluation Dataset: 100%|██████████| 4/4 [00:18<00:00,  4.53s/it]

  multi_turn_task_success_v1         1.25/5  [FAIL]  (errors=0/4)



Loss patterns found: 1  (feed the biggest into Phase 4 below)

▸ Instruction Following › Over-Punting  — 1 case(s)
    The agent explicitly declines the task, claiming it lacks a tool or information it actually possesses.
      ✗ expected: The response confirms that the entertainment expense limit has been successfully raised to $9,000 and approved.
        why: The agent explicitly declined the user's request, claiming it lacked the capability to modify or approve expense limits ("I'm unable to directly modify or approve expense limits myself"). This is a failure to fulfill the request despite having access to a specialized `expense_agent`, which characterizes over-punting.


## Phase 4 — Optimize the agent with the ADK optimizer (optimize-agent) — close the flywheel 🔁
📖 [ADK optimize](https://adk.dev/optimize/) · [optimize-agent](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/optimize-agent)

We re-tune the coordinator's instruction with the **ADK optimizer** — the inline
[adk.dev/optimize](https://adk.dev/optimize/) flow: `GEPARootAgentPromptOptimizer` (evolutionary prompt
search, google-adk ≥ 1.24; this repo runs 2.5), *not* the Vertex prompt optimizer. A `LocalEvalSampler`
scores each candidate instruction against the coordinator's ADK evalset
(`src/agents/coordinator/coordinator_eval_set.evalset.json`) with the deterministic `response_match_score`
metric; GEPA keeps the best-scoring rewrite.

**Feeding Phase 4a's clusters in.** `optimize(agent, sampler)` takes no *clusters* argument, so we
warm-start GEPA from a **cluster-annotated instruction**: the `LOSS_GUIDANCE` string Phase 4a produced
(each loss pattern as a "fix this failure mode" note) is appended to the root instruction GEPA evolves,
and GEPA also sees the failing cases directly through the sampler. (Alternative: turn the clustered
failure prompts into new eval cases — but those need reference answers for `response_match_score`.)

**Live job**: it runs the agent (MCP tools must be reachable) + models — a few minutes at the small demo
budget below (production default is 100 metric calls ≈ 10–20 min); it rewrites only the root agent's
instruction. The cell uses notebook top-level `await`; set `GEAP_SKIP_OPT=1` to skip (e.g. in CI).

In [14]:
# --- ADK optimizer (adk.dev/optimize) — inline GEPA; NOT the Vertex optimizer ---
# Feeds Phase 4a's loss clusters in by WARM-STARTING the instruction GEPA evolves; GEPA also sees the
# failing cases via the sampler. Live job — the optimizer runs the agent LOCALLY, so its MCP tools
# must be reachable from here. Set GEAP_SKIP_OPT=1 to skip.
import os

if os.environ.get("GEAP_SKIP_OPT") == "1":
    print("Skipping optimization (GEAP_SKIP_OPT=1).  Run it with:")
    print("  uv run python -m src.optimize.run_optimize src/agents/coordinator --optimizer gepa")
else:
    # The coordinator's tools come from remote MCP servers. Reach them by their direct HTTPS URLs
    # (from config) instead of the Agent Registry, whose mTLS handshake isn't available here and would
    # leave the toolsets empty ("Tool '...' not found. Available tools: transfer_to_agent"). These env
    # vars MUST be set before importing the agent module (it reads them at import time).
    os.environ["MCP_USE_DIRECT_URLS"] = "1"
    from src.config import SEARCH_MCP_URL, BOOKING_MCP_URL, EXPENSE_MCP_URL
    os.environ.setdefault("SEARCH_MCP_URL", SEARCH_MCP_URL)
    os.environ.setdefault("BOOKING_MCP_URL", BOOKING_MCP_URL)
    os.environ.setdefault("EXPENSE_MCP_URL", EXPENSE_MCP_URL)
    # Route ADK/genai model calls through Vertex (ADC) — GEPA's sampler + reflection models need this.
    os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "true")
    os.environ.setdefault("GOOGLE_CLOUD_PROJECT", GCP_PROJECT_ID)
    os.environ.setdefault("GOOGLE_CLOUD_LOCATION", GCP_REGION)

    from google.adk.evaluation.eval_config import EvalConfig
    from google.adk.evaluation.local_eval_sets_manager import LocalEvalSetsManager
    from google.adk.optimization.gepa_root_agent_prompt_optimizer import (
        GEPARootAgentPromptOptimizer, GEPARootAgentPromptOptimizerConfig)
    from google.adk.optimization.local_eval_sampler import LocalEvalSampler, LocalEvalSamplerConfig
    from src.agents.coordinator.agent import root_agent as coordinator

    # Quiet ADK/MCP/auth logs during the run (recovered connection churn is noise; the summary below
    # is what matters). Re-asserted here in case importing ADK reset the levels from the setup cell.
    import logging as _lg
    for _n in ("google_adk", "google.adk", "mcp", "google.auth", "grpc"):
        _lg.getLogger(_n).setLevel(_lg.CRITICAL)

    # 🔁 Feed the clusters in: warm-start from a cluster-annotated instruction (LOSS_GUIDANCE is set by
    # Phase 4a). model_copy leaves the deployed module untouched; sub-agents/tools are preserved.
    guidance = globals().get("LOSS_GUIDANCE", "") or ""
    initial_agent = coordinator
    if guidance:
        initial_agent = coordinator.model_copy(update={
            "instruction": coordinator.instruction
            + "\n\n# Known failure modes to fix (from Phase 4a loss-cluster analysis):\n" + guidance})
        print("Warm-starting GEPA with Phase 4a loss patterns:\n" + guidance + "\n")
    else:
        print("No LOSS_GUIDANCE found (run Phase 4a first) — optimizing the base instruction.\n")

    # Sampler: score each candidate instruction against the coordinator's ADK evalset with the
    # deterministic response_match_score (LLM-judge metrics can return None and crash GEPA's rounding).
    sampler = LocalEvalSampler(
        LocalEvalSamplerConfig(
            eval_config=EvalConfig(criteria={"response_match_score": 0.5}),
            app_name="coordinator",                  # = the src/agents/coordinator dir name
            train_eval_set="coordinator_eval_set"),  # coordinator_eval_set.evalset.json
        LocalEvalSetsManager(agents_dir="src/agents"))

    # GEPA — bounded for a demo (default is 100 metric calls ≈ 10-20 min).
    opt_config = GEPARootAgentPromptOptimizerConfig(
        max_metric_calls=int(os.environ.get("GEAP_MAX_METRIC_CALLS", "20")))
    optimizer = GEPARootAgentPromptOptimizer(config=opt_config)

    try:
        # Notebook top-level await (Jupyter/Colab/nbconvert) — asyncio.run() would clash with the
        # kernel's running event loop. In a plain .py script use asyncio.run(optimizer.optimize(...)).
        result = await optimizer.optimize(initial_agent, sampler)
        best = result.optimized_agents[result.gepa_result["best_idx"]]
        print("Validation score:", best.overall_score)
        print("\nOptimized coordinator instruction:\n" + "-" * 80)
        print(best.optimized_agent.instruction)
        print("-" * 80)
        print("\n✓ Paste this into src/agents/coordinator/agent.py, redeploy, then re-run Phases 2a/4a —")
        print("  if the loss patterns shrank, the flywheel turned. 🔁")
    except Exception as e:
        print(f"ADK optimization didn't finish here ({type(e).__name__}: {e}).")
        print("It needs reachable MCP tool servers + Vertex ADC. From a shell:")
        print("  uv run python -m src.optimize.run_optimize src/agents/coordinator --optimizer gepa")

Warm-starting GEPA with Phase 4a loss patterns:
- Avoid 'Instruction Following › Over-Punting': The agent explicitly declines the task, claiming it lacks a tool or information it actually possesses.



Iteration 0: Base program full valset score: 0.5882352941176471 over 17 / 17 examples
Validation score: 0.5882352941176471

Optimized coordinator instruction:
--------------------------------------------------------------------------------
You are a corporate assistant coordinator. Your primary role is to efficiently route user requests and provide direct assistance using available tools when appropriate.

1. Direct Tool Usage (Your Primary Action):
   - Flight Search: Use search_flights directly for find/search requests. If invalid airport codes are returned, inform the user clearly.
   - Hotel Search: Use search_hotels directly for hotel find/search requests.
   - Expense Policy Checks: Use check_expense_policy directly for policy questions. Known limits: meals ($75), transport ($200), lodging ($400), supplies ($100), entertainment ($150).
   - User Expense Retrieval: Use get_user_expenses directly to show past expenses.

2. Delegation (Transfer to Specialist Agent):
   - Flight/Hote

## Quality-drift alerts (quality-alerts) — 🔧 custom
📖 [quality-alerts](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/quality-alerts)

Configuring a GEAP **Online Monitor** auto-exports numeric eval scores to the Cloud Monitoring metric `aiplatform.googleapis.com/online_evaluator/scores` — a DELTA **distribution** on the `aiplatform.googleapis.com/OnlineEvaluator` resource, labelled by `evaluation_metric_name`. A sustained drop in the median score signals **quality drift**. Here we build and create the alert policy directly with the `monitoring_v3` API (the same call `src/eval/quality_alerts.py` wraps); the commented tail shows the GitOps alternative — render the policy to YAML and apply it with `gcloud`.

In [15]:
# 🔧 CUSTOM: build + create the drift alert with the Cloud Monitoring API (not just a printout).
from google.cloud import monitoring_v3
from google.protobuf import duration_pb2
from src.config import GCP_PROJECT_ID

METRIC_NAME  = "task_success"   # the online-evaluator metric to watch
THRESHOLD    = 0.8              # page when the median score drops below this
DISPLAY_NAME = f"GEAP Agent Quality Drift - Low {METRIC_NAME}"

alert_client = monitoring_v3.AlertPolicyServiceClient()
project = f"projects/{GCP_PROJECT_ID}"

# online_evaluator/scores is a DELTA DISTRIBUTION on the OnlineEvaluator resource, so the
# filter MUST restrict resource.type and the aligner must be a percentile (ALIGN_MEAN is
# rejected for distributions) — ALIGN_PERCENTILE_50 is the median score in the window.
condition = monitoring_v3.AlertPolicy.Condition(
    display_name=f"{METRIC_NAME} median online_evaluator score below {THRESHOLD}",
    condition_threshold=monitoring_v3.AlertPolicy.Condition.MetricThreshold(
        filter=(
            'resource.type="aiplatform.googleapis.com/OnlineEvaluator" '
            'AND metric.type="aiplatform.googleapis.com/online_evaluator/scores" '
            f'AND metric.labels.evaluation_metric_name="{METRIC_NAME}"'
        ),
        comparison=monitoring_v3.ComparisonType.COMPARISON_LT,
        threshold_value=THRESHOLD,
        duration=duration_pb2.Duration(seconds=3600),            # must hold for 60 min
        aggregations=[monitoring_v3.Aggregation(
            alignment_period=duration_pb2.Duration(seconds=3600),
            per_series_aligner=monitoring_v3.Aggregation.Aligner.ALIGN_PERCENTILE_50,
        )],
    ),
)

policy = monitoring_v3.AlertPolicy(
    display_name=DISPLAY_NAME,
    documentation=monitoring_v3.AlertPolicy.Documentation(
        content=(f"The agent's '{METRIC_NAME}' online-eval median score dropped below "
                 f"{THRESHOLD} — it is regressing vs its evaluated baseline."),
        mime_type="text/markdown"),
    conditions=[condition],
    combiner=monitoring_v3.AlertPolicy.ConditionCombinerType.OR,
    # notification_channels=["projects/.../notificationChannels/<ID>"],  # page a channel
    enabled=True,
)

# Idempotent create: reuse an existing same-named policy so re-running the notebook is safe.
existing = [p for p in alert_client.list_alert_policies(name=project)
            if p.display_name == DISPLAY_NAME]
if existing:
    print(f"\u2713 Alert policy already exists: {existing[0].name}")
else:
    created = alert_client.create_alert_policy(name=project, alert_policy=policy)
    print(f"\u2713 Alert policy created: {created.name}")
print(f"  Condition: median online_evaluator/scores[{METRIC_NAME}] < {THRESHOLD} (60-min window)")

# GitOps alternative — render the identical policy to YAML and apply it with gcloud:
#   from src.eval.quality_alerts import export_policy_yaml
#   path = export_policy_yaml("src/eval/policies/quality_drift_policy.yaml")
#   gcloud monitoring policies create --policy-from-file=src/eval/policies/quality_drift_policy.yaml

✓ Alert policy already exists: projects/wortz-project-352116/alertPolicies/2588083795864489538
  Condition: median online_evaluator/scores[task_success] < 0.8 (60-min window)


## Recap — L1 SDK vs. acknowledged custom code

**Pure L1 SDK (`client.evals.*` / `vertexai.types.*`):**
- `client.evals.evaluate(dataset, metrics)` — rapid, regression, simulated, and offline scoring
- `client.evals.generate_conversation_scenarios(...)` — synthetic multi-turn scenarios
- `client.evals.run_inference(agent=…)` → `generate_loss_clusters(eval_result, metric)` — Automatic Loss Analysis: replay prompts to capture agent trajectories, then cluster failures into loss patterns (**global** region)
- `client.evals.create_evaluation_metric(...)` / `list_evaluation_metrics()` — Metric Registry (publish + reuse)
- metrics: `types.RubricMetric.*`, `types.LLMMetric` + `types.MetricPromptBuilder`, `types.CodeExecutionMetric`
- data: `types.EvaluationDataset`, `types.EvalCase`, `types.ResponseCandidate`, `types.evals.AgentInfo`
- `result.show()` — view-results tables (aggregate + per-case + loss clusters)

**🔧 Acknowledged custom (not eval-SDK, or SDK-bug workarounds):**
- inference via `agent_engines.stream_query` in Phases 2a–2e (Phase 4a instead uses `client.evals.run_inference`, which returns `agent_data` and — needed for loss clustering — works on the current build)
- resilience / environment simulation (MCP tool mocking + fault injection)
- reading historical OTel traces from BigQuery
- online-monitor setup (`setup_online_evaluators.py`)
- ADK optimizer (Phase 4) — `GEPARootAgentPromptOptimizer` / `SimplePromptOptimizer` ([adk.dev/optimize](https://adk.dev/optimize/)); the repo also wraps it in `src/optimize/run_optimize.py`
- publishing eval scores to Cloud Monitoring (`create_time_series`) + pulling them from Metrics Explorer (`list_time_series`)
- Cloud Monitoring alert policy (`monitoring_v3` / gcloud)

Headless orchestrator + JSON report:
`uv run python -m src.eval.demo.full_eval_demo --agent-id $AGENT_ENGINE_ID`